In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import sys
import os

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)

In [ ]:
from package_files.benefits_defns import *

In [ ]:
path = '../../data/salary_sample_body_benefits_v3.parquet.gzip'

In [ ]:
if path[-3:] == 'csv:':
    even_sample = pd.read_csv(path)
elif path[-3:] == 'zip':
    even_sample = pd.read_parquet(path)

In [ ]:
even_sample

In [ ]:
even_sample.columns

In [ ]:
pd.set_option('display.max_rows', 300)
even_sample.groupby([occupation, 'YEAR','AI ROLE']).size()

In [ ]:
occupations_select = ['Architecture and Engineering Occupations','Arts, Design, Entertainment, Sports, and Media Occupations','Business and Financial Operations Occupations','Community and Social Service Occupations','Computer and Mathematical Occupations',
'Educational Instruction and Library Occupations',
'Healthcare Practitioners and Technical Occupations',
'Legal Occupations',
'Life, Physical, and Social Science Occupations',
'Management Occupations', 
'Office and Administrative Support Occupations',
'Personal Care and Service Occupations', 'Production Occupations',
'Sales and Related Occupations',
'Transportation and Material Moving Occupations',
]

In [ ]:
len(occupations_select)

In [ ]:
even_sample_select = even_sample[even_sample[occupation].isin(occupations_select)]

# AI Role Demand Over Time

In [ ]:
ai_role_raw = even_sample_select.groupby([occupation, 'YEAR'])['AI ROLE'].sum().reset_index()

In [ ]:
role_total = even_sample_select.groupby([occupation, 'YEAR'])['AI ROLE'].count().reset_index()

In [ ]:
role_total

In [ ]:
ai_role_raw

In [ ]:
ai_role_occupation = even_sample_select.groupby([occupation, 'YEAR'])['AI ROLE'].mean().reset_index()


In [ ]:
ai_role_occupation.rename(columns={'AI ROLE':'AI ROLE %'}, inplace=True)

In [ ]:
ai_role_occupation

In [ ]:
import matplotlib.pyplot as plt

# use different colors for each occupation
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan', 'black', 'yellow', 'lime', 'teal', 'magenta']

# Create a pivot table where the index is 'YEAR' and columns are 'occupation'
pivot_data = ai_role_occupation.pivot(index='YEAR', columns=occupation, values='AI ROLE %')

# Plot each occupation's percentage of AI roles over time
pivot_data.plot(figsize=(10, 6), marker='o', color=colors)

# Adding titles and labels
plt.title('Percentage of AI Roles Over Time by Occupation')
# make x ticks only year integers
plt.xticks(np.arange(pivot_data.index.min(), pivot_data.index.max()+1, 1.0))
plt.xlabel('Year')
plt.ylabel('Percentage of AI Roles')
plt.legend(title='Occupation', bbox_to_anchor=(1, 1))
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
pivot_data

# % Roles Offering Benefit for AI/Non-AI Roles

In [ ]:
occupation_benefits = even_sample_select.groupby([occupation, 'YEAR', 'AI ROLE'])[benefits3].mean().reset_index()

In [ ]:
occupation_benefits_all = even_sample_select.groupby([occupation, 'YEAR'])[(benefits3 + ['LOG_SALARY'])].mean().reset_index()

In [ ]:
occupation_benefits_all.rename(columns={'LOG_SALARY':'MEAN_LOG_SALARY'}, inplace=True)

In [ ]:
pd.set_option('display.max_rows', 20)

In [ ]:
occupation_benefits_all

In [ ]:
occupation_benefits

In [ ]:
benefits_ai = occupation_benefits[occupation_benefits['AI ROLE'] == 1]
benefits_non_ai = occupation_benefits[occupation_benefits['AI ROLE'] == 0]

In [ ]:
merged_df = pd.merge(benefits_ai, benefits_non_ai, on=['SOC_2021_2_NAME', 'YEAR'], suffixes=('_ai', '_non_ai'))


In [ ]:
for benefit in benefits3:
    merged_df[f'{benefit}_premium'] = ((merged_df[f'{benefit}_ai'] - (merged_df[f'{benefit}_non_ai']
                                                                      ))*100).round(3)


In [ ]:
for benefit in benefits3:
    merged_df[f'{benefit}_premium'] = merged_df[f'{benefit}_premium'].apply(lambda x: f'{x:.3f}' if pd.notnull(x) else x)

In [ ]:
pd.set_option('display.max_columns', 300)

In [ ]:
merged_df

In [ ]:
benefit_premium = merged_df[[occupation, 'YEAR'] + [f'{benefit}_premium' for benefit in benefits3]]

In [ ]:
benefit_premium

In [ ]:
# set premium columns as float in benefit_premium
for benefit in benefits3:
    benefit_premium[f'{benefit}_premium'] = benefit_premium[f'{benefit}_premium'].astype(float)

In [ ]:
# group benefits_premium by occupation, not year and get mean
# don't include year column in groupby
benefit_premium_grouped = benefit_premium.groupby(occupation).mean().reset_index()
benefit_premium_grouped.drop(columns='YEAR', inplace=True)

In [ ]:
benefit_premium_grouped

In [ ]:
import matplotlib.pyplot as plt

# use different colors for each occupation
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan', 'black', 'yellow', 'lime', 'teal', 'magenta']

for i, benefit in enumerate(benefits3):
    
    # Create a pivot table where the index is 'YEAR' and columns are 'occupation'
    pivot_data = benefit_premium.pivot(index='YEAR', columns=occupation, values=f'{benefit}_premium')

    # Plot each occupation's percentage of AI roles over time
    pivot_data.plot(figsize=(10, 6), marker='o', color=colors)
    benefit_label = benefits3_labels[i]
    # Adding titles and labels
    plt.title(f'{benefit_label} Premium Over Time by Occupation')
    # make x ticks only year integers
    plt.xticks(np.arange(pivot_data.index.min(), pivot_data.index.max()+1, 1.0))
    plt.xlabel('Year')
    plt.ylabel('Premium')
    # plt.legend(None)
    plt.grid(True)
    plt.tight_layout()
    plt.legend().remove()
    plt.savefig(f'../figures/{benefit}_premium_time.png')

    # Show the plot
    plt.show()


In [ ]:
occ_year_df = benefit_premium.merge(ai_role_occupation, on=[occupation, 'YEAR'])

In [ ]:
occ_year_df

# Monetary Premium

In [ ]:
occupation_salaries = occupation_benefits = even_sample_select.groupby([occupation, 'YEAR', 'AI ROLE'])[['LOG_SALARY', 'SALARY']].mean().reset_index()

In [ ]:
occupation_salaries_ai = occupation_salaries[occupation_salaries['AI ROLE'] == 1]
occupation_salaries_non_ai = occupation_salaries[occupation_salaries['AI ROLE'] == 0]

In [ ]:
occupation_salaries = pd.merge(occupation_salaries_ai, occupation_salaries_non_ai, on=[occupation, 'YEAR'], suffixes=('_ai', '_non_ai'))

In [ ]:
occupation_salaries['SALARY_PREMIUM_LOG'] = ((occupation_salaries['LOG_SALARY_ai']/occupation_salaries['LOG_SALARY_non_ai'])*100)

In [ ]:
occupation_salaries['SALARY_PREMIUM'] = ((occupation_salaries['SALARY_ai']/occupation_salaries['SALARY_non_ai'])*100)

In [ ]:
occupation_salaries

In [ ]:
occupation_salaries['SALARY_PREMIUM'].hist()
plt.title('Salary Premium Distribution')

# Change in Demand

In [ ]:
ai_role_occupation = ai_role_occupation.sort_values(by=[occupation, 'YEAR'])

In [ ]:
ai_role_occupation['AI ROLE % CHANGE'] = ai_role_occupation.groupby(occupation)['AI ROLE %'].pct_change()

In [ ]:
ai_role_occupation['AI ROLE % CHANGE'] = ai_role_occupation['AI ROLE % CHANGE']*100

In [ ]:
ai_role_occupation

In [ ]:
ai_role_occupation['PRIOR YEAR % CHANGE'] = ai_role_occupation.groupby(occupation)['AI ROLE % CHANGE'].shift(1)

In [ ]:
ai_role_occupation

In [ ]:
occ_year_df = occ_year_df.merge(ai_role_occupation, on=[occupation, 'YEAR'])

In [ ]:
occ_year_df

In [ ]:
occ_year_df.drop(columns='AI ROLE %_y', inplace=True)

In [ ]:
occ_year_df.rename(columns={'AI ROLE %_x':'AI ROLE %'}, inplace=True)

In [ ]:
occ_year_df

In [ ]:
occ_year_df = occ_year_df.merge(occupation_salaries, on=[occupation, 'YEAR'])

In [ ]:
occ_year_df.drop(columns=['AI ROLE_ai', 'AI ROLE_non_ai'], inplace=True)

# Scatterplots

In [ ]:
benefits3

## Vs AI Demand

In [ ]:
# scatterplot of CAREER PREMIUM vs AI ROLE %
import seaborn as sns
for i, benefit in enumerate(benefits3):
    plt.scatter(occ_year_df[f'{benefit}_premium'], occ_year_df['AI ROLE %'], c=occ_year_df[year])
    plt.xlabel(f'{benefits3_labels[i]} Premium')
    plt.ylabel('AI Role %')
    plt.show()


## Color by Occupation

In [ ]:
for i, benefit in enumerate(benefits3):
    plt.figure(figsize=(8, 6))
    # Remove outliers for each benefit's premium and 'AI ROLE %'
    # filtered_df = remove_outliers(occ_year_df, f'{benefit}_premium')
    # filtered_df = remove_outliers(filtered_df, 'AI ROLE %')
    # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
    sns.scatterplot(
        data=occ_year_df, 
        x=f'{benefit}_premium', 
        y='AI ROLE %', 
        hue='SOC_2021_2_NAME',  # Color by occupation
        palette='Set1',         # Optional: Choose a color palette
        legend=False
    )
    
    # Set axis labels and title
    plt.xlabel(f'{benefits3_labels[i]} Premium')
    plt.ylabel('AI Role %')
    plt.title(f'{benefits3_labels[i]} Premium vs AI Role % by Occupation')
    plt.savefig(f'../figures/scatter_premium_vs_ai_demand_{benefit}.png')
    
    # Show the plot
    plt.show()

### Without Outliers

In [ ]:
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    # Define outlier boundaries
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Iterate through each benefit and plot the scatterplot without outliers
for i, benefit in enumerate(benefits3):
    # Remove outliers for each benefit's premium and 'AI ROLE %'
    filtered_df = remove_outliers(occ_year_df, f'{benefit}_premium')
    filtered_df = remove_outliers(filtered_df, 'AI ROLE %')
    
    # Plot the scatterplot after removing outliers
    plt.scatter(filtered_df[f'{benefit}_premium'], filtered_df['AI ROLE %'], c=filtered_df[year])
    plt.xlabel(f'{benefits3_labels[i]} Premium')
    plt.ylabel('AI Role %')
    plt.title(f'{benefits3_labels[i]} Premium vs AI Role %')
    plt.show()

## Change in Demand

In [ ]:
for i, benefit in enumerate(benefits3):
    plt.figure(figsize=(8, 6))
    
    # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
    sns.scatterplot(
        data=occ_year_df, 
        x=f'{benefit}_premium', 
        y='AI ROLE % CHANGE', 
        hue='SOC_2021_2_NAME',  # Color by occupation
        palette='Set1',         # Optional: Choose a color palette
        legend=False
    )
    
    # Set axis labels and title
    plt.xlabel(f'{benefits3_labels[i]} Premium')
    plt.ylabel('AI Role % Change')
    plt.title(f'{benefits3_labels[i]} Premium vs AI Role % Change')
    plt.savefig(f'../figures/scatter_premium_vs_ai_demand_change_{benefit}.png')
    
    # Show the plot
    plt.show()

## AI Role Prior Year Change

In [ ]:
for i, benefit in enumerate(benefits3):
    plt.figure(figsize=(8, 6))
    
    # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
    sns.scatterplot(
        data=occ_year_df, 
        x=f'{benefit}_premium', 
        y='PRIOR YEAR % CHANGE', 
        hue='SOC_2021_2_NAME',  # Color by occupation
        palette='Set1',         # Optional: Choose a color palette
        legend=False
    )
    
    # Set axis labels and title
    plt.xlabel(f'{benefits3_labels[i]} Premium')
    plt.ylabel('Prior Year AI % Change')
    plt.title(f'{benefits3_labels[i]} Premium vs Prior Year AI % Change')
    plt.savefig(f'../figures/scatter_premium_prior_year_change_{benefit}.png')
    
    # Show the plot
    plt.show()

# Vs. Salary Premium

In [ ]:
for i, benefit in enumerate(benefits3):
    plt.figure(figsize=(8, 6))
    
    # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
    sns.scatterplot(
        data=occ_year_df, 
        x=f'{benefit}_premium', 
        y='SALARY_PREMIUM', 
        hue='SOC_2021_2_NAME',  # Color by occupation
        palette='Set1',         # Optional: Choose a color palette
        legend=False
    )
    
    # Set axis labels and title
    plt.xlabel(f'{benefits3_labels[i]} Premium')
    plt.ylabel('Monetary Premium')
    plt.title(f'{benefits3_labels[i]} Premium vs Monetary Premium')
    plt.savefig(f'../figures/scatter_premium_salary_{benefit}.png')
    
    # Show the plot
    plt.show()

## Vs. Log Salary Premium

In [ ]:
occ_year_df.columns

In [ ]:
for i, benefit in enumerate(benefits3):
    plt.figure(figsize=(8, 6))
    filtered_df = remove_outliers(occ_year_df, f'{benefit}_premium')
    filtered_df = remove_outliers(filtered_df, 'SALARY_PREMIUM_LOG')
    
    # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
    sns.scatterplot(
        data=filtered_df, 
        x=f'{benefit}_premium', 
        y='SALARY_PREMIUM_LOG', 
        hue='SOC_2021_2_NAME',  # Color by occupation
        palette='Set1',         # Optional: Choose a color palette
        legend=False
    )
    
    # Set axis labels and title
    plt.xlabel(f'{benefits3_labels[i]} Premium')
    plt.ylabel('Monetary Premium (Log Salary)')
    plt.title(f'{benefits3_labels[i]} Premium vs Monetary Premium (Log Salary)')
    plt.savefig(f'../figures/scatter_premium_log_salary_{benefit}.png')
    
    # Show the plot
    plt.show()

# % Benefits Vs. Salary

In [ ]:
occ_year_df

In [ ]:
len(occ_year_df)

In [ ]:
mean_salary = even_sample_select.groupby([occupation, 'YEAR'])[['SALARY', 'LOG_SALARY']].mean().reset_index()

In [ ]:
occ_year_df = occ_year_df.merge(mean_salary, on=[occupation, 'YEAR'])

In [ ]:
pd.set_option('display.max_rows', 100)

In [ ]:
occ_year_df.rename(columns={'SALARY':'MEAN SALARY'}, inplace=True)

In [ ]:
occ_year_df.drop(columns='SALARY', inplace=True)

In [ ]:
occ_year_df.rename(columns={'LOG_SALARY':'MEAN LOG SALARY'}, inplace=True)

In [ ]:
occupation_benefits_all.columns

In [ ]:
for i, benefit in enumerate(benefits3):
    plt.figure(figsize=(8, 6))
    # filtered_df = remove_outliers(occupation_benefits_all, f'{benefit}_premium')
    correlation = occupation_benefits_all[[f'{benefit}', 'MEAN_LOG_SALARY']].corr().iloc[0, 1]

    # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
    sns.scatterplot(
        data=occupation_benefits_all, 
        x=f'{benefit}', 
        y='MEAN_LOG_SALARY', 
        hue='SOC_2021_2_NAME',  # Color by occupation
        palette='Set1',         # Optional: Choose a color palette
        legend=False
    )
    
    # Set axis labels and title
    plt.xlabel(f'% Jobs Offering {benefits3_labels[i]} Benefits')
    plt.ylabel('Mean Log Salary')
    plt.title(f'{benefits3_labels[i]} vs Mean Log Salary\nCorrelation: {correlation:.3f}')
    plt.savefig(f'../figures/scatter_log_salary_{benefit}_all.png')
    
    # Show the plot
    plt.show()

# Models

In [ ]:
import sys
import os

# Get the absolute path to the folder containing my_module.py
module_path = os.path.abspath(os.path.join('..'))

# Add that directory to sys.path
sys.path.append(module_path)

In [ ]:
from logit_model import *

In [ ]:
occ_year_df.columns

In [ ]:
# run linear regression model 
occ_models_demand = []
for benefit in benefits3:
    X = occ_year_df['AI ROLE %'] 
    y = occ_year_df[benefit + '_premium']
    # Fit the linear regression model
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    occ_models_demand.append(model)
    print(model.summary())




In [ ]:
occ_year_df['AI ROLE % CHANGE'].value_counts(dropna=False)

In [ ]:
# run linear regression model 
occ_models_demand_change = []
for benefit in benefits3:
    # drop nas from AI ROLE % CHANGE
    model_df = occ_year_df.dropna(subset=['AI ROLE % CHANGE'])
    X = model_df['AI ROLE % CHANGE'] 
    y = model_df[benefit + '_premium']
    # Fit the linear regression model
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    occ_models_demand_change.append(model)
    print(model.summary())




In [ ]:
occ_models_demand_change_prior = []
for benefit in benefits3:
    # drop nas from PRIOR YEAR % CHANGE
    model_df = occ_year_df.dropna(subset=['PRIOR YEAR % CHANGE'])
    X = model_df['PRIOR YEAR % CHANGE'] 
    y = model_df[benefit + '_premium']
    # Fit the linear regression model
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    occ_models_demand_change_prior.append(model)
    print(model.summary())

In [ ]:
occ_models_monetary_premium = []
for benefit in benefits3:
    # drop nas from PRIOR YEAR % CHANGE
    model_df = occ_year_df.dropna(subset=['SALARY_PREMIUM_LOG'])
    X = model_df['SALARY_PREMIUM_LOG'] 
    y = model_df[benefit + '_premium']
    # Fit the linear regression model
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    occ_models_monetary_premium.append(model)
    print(model.summary())

In [ ]:
def create_results_df(model_list, benefit_list, variable, description):
    coefficients = []
    errors = []
    pvalues = []
    for model in model_list:
        try:
            coef = model.params[variable]
            err = model.bse[variable]
            pvalue = model.pvalues[variable].round(3)
        except:
            coef = None
            err = None
            pvalue = None

        coefficients.append(coef)
        errors.append(err)
        pvalues.append(pvalue)

    # Creating DataFrame
    results = {
        'Label': benefit_list,
        'Coefficient': coefficients,
        'Error': errors,
        'P-Value': pvalues
    }

    results_df = pd.DataFrame(results)
    results_df['Model Iteration'] = description
    return results_df

In [ ]:
def plot_model_results(results_df, benefits_list):
    # Define colors for each model
    colors = ['#E69F00', '#56B4E9', '#009E73', '#CC79A7']
    # Calculate the 95% confidence intervals
    results_df['Lower_CI'] = results_df['Coefficient'] - 1.96 * results_df['Error']
    results_df['Upper_CI'] = results_df['Coefficient'] + 1.96 * results_df['Error']

    # Plotting
    fig, ax = plt.subplots(figsize=(12, 6))

    # benefits = benefits
    model_iterations = results_df['Model Iteration'].unique()
    markers = ['o', 's', '^', 'D']  # Different markers for model iterations
    positions = []
    current_pos = 0

    # Store legend handles and labels to avoid duplicates
    handles, labels = [], []

    for benefit in benefits_list:
        benefit_data = results_df[results_df['Label'] == benefit]
        benefit_positions = []
        for i, model in enumerate(model_iterations):
            model_data = benefit_data[benefit_data['Model Iteration'] == model]
            if not model_data.empty:
                pos = current_pos + i * 0.2  # Adjust spacing between model_iterations within the same benefit
                handle = ax.errorbar(
                    pos, model_data['Coefficient'].values, 
                    yerr=[model_data['Coefficient'].values - model_data['Lower_CI'].values, 
                        model_data['Upper_CI'].values - model_data['Coefficient'].values], 
                    fmt=markers[i], color=colors[i], label=model if benefit == benefits_list[0] else ""
                )
                benefit_positions.append(pos)
                if benefit == benefits_list[0]:  # Add handles and labels only for the first benefit to avoid duplicates
                    handles.append(handle)
                    labels.append(model)
        positions.extend(benefit_positions)
        current_pos += len(model_iterations) + 1  # Add more space between different benefits

    # Customize plot
    benefit_ticks = [(positions[i * len(model_iterations)] + positions[(i + 1) * len(model_iterations) - 1]) / 2 for i in range(len(benefits_list))]
    ax.set_xticks(benefit_ticks, labels = benefits3_labels)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_xticklabels(benefits3_labels, rotation=45, fontsize = 14, ha='right')
    # ax.set_title('AI Skill Coefficients by benefit and Occupation')
    ax.set_xlabel(None)
    ax.set_ylabel('Log-Odds Coefficient (with 95% CI)', fontsize=16)
    ax.axhline(0, color='grey', linewidth=0.8)
    ax.legend(handles, labels, title='Model', bbox_to_anchor=(0, 1), loc = 'upper left', fontsize = 12, title_fontsize = 12)
    # ax.axhline(average_coefficient, color='black', linestyle='--', linewidth=1, label='Average Coefficient')
    plt.title('Occupation-Year OLS Models \n(Dependent Variable: Benefit Premium)', fontsize=18)
    plt.tight_layout()
    plt.savefig('../figures/occ_model_coefficients_plot.png')
    plt.show()


In [ ]:
results_demand = create_results_df(occ_models_demand, benefits3, 'AI ROLE %', 'AI Demand')

In [ ]:
results_demand

In [ ]:
results_demand_change = create_results_df(occ_models_demand_change, benefits3, 'AI ROLE % CHANGE', 'AI Demand Change')

In [ ]:
results_demand_change_prior = create_results_df(occ_models_demand_change_prior, benefits3, 'PRIOR YEAR % CHANGE', 'AI Demand Change Prior Year')

In [ ]:
results_monetary_premium = create_results_df(occ_models_monetary_premium, benefits3, 'SALARY_PREMIUM_LOG', 'Monetary Premium (Log Salary)')

In [ ]:
results_df = pd.concat([results_demand, results_demand_change, results_demand_change_prior, results_monetary_premium])

In [ ]:
plot_model_results(results_df, benefits3)

# Benefits by Occupation & Role Type

In [ ]:
even_sample = pd.read_parquet('../../data/salary_sample_2018_2023.parquet.gzip')

In [ ]:
even_sample_select = even_sample[even_sample[occupation].isin(occupations_select)]

In [ ]:
def get_benefit_by_occupation(even_sample, benefit):
    occ_ai_benefit_group = even_sample.groupby([occupation, 'AI ROLE',benefit]).size().reset_index(name='benefit_count')
    occ_ai_group = even_sample.groupby([occupation, 'AI ROLE']).size().reset_index(name='count')
    occ_ai_benefit_group = occ_ai_benefit_group.merge(occ_ai_group, on = ['SOC_2021_2_NAME','AI ROLE'], how = 'left')
    occ_ai_benefit_group[f'Percent with {benefit}'] = occ_ai_benefit_group['benefit_count']/occ_ai_benefit_group['count']
    occ_ai_benefit_percent = occ_ai_benefit_group[occ_ai_benefit_group[benefit] == 1.0].copy()
    # rename columns without setting copy of a slice
    
    
    occ_ai_benefit_percent.rename(columns = {'SOC_2021_2_NAME': 'Occupation'}, inplace=True)
    occ_ai_benefit_percent['AI ROLE'] = occ_ai_benefit_percent['AI ROLE'].map({True: 'Yes', False: 'No'})
    occ_ai_benefit_percent['Occupation'] = occ_ai_benefit_percent['Occupation'].str.replace(' Occupations', '')

    return occ_ai_benefit_percent
    

In [ ]:
benefit_percent_dfs = []
for benefit in benefits4:
    occ_ai_benefit_percent = get_benefit_by_occupation(even_sample_select, benefit)
    benefit_percent_dfs.append(occ_ai_benefit_percent)

In [ ]:
occ_ai_benefit_percent

In [ ]:
def plot_benefit_occ_percents(df, benefit):
    sorted_data = df[df['AI ROLE'] == 'No']
    sorted_data = sorted_data.sort_values(by=f'Percent with {benefit}', ascending=True)

    # Create a figure and axis object
    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot the data as a clustered bar chart
    sns.barplot(x=f'Percent with {benefit}', y='Occupation', hue='AI ROLE', data=df, ax=ax)

    # Set the labels and title
    ax.set_xlabel(f'Percent Jobs with {benefits_labels_map[benefit]}', fontsize=14)
    ax.set_ylabel('Occupation', fontsize=14)
    ax.set_title(None)

    # Add value labels without decimals
    # for p in ax.patches:
    #     width = p.get_width()
    #     ax.annotate(f'{width:.1f}%', (p.get_x() + p.get_width(), p.get_y() + p.get_height() / 2.),
    #                 ha='center', va='center', fontsize=12, color='black', xytext=(5, 0),
    #                 textcoords='offset points')

    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45)

    # Adjust the legend position and title
    plt.legend(title='AI ROLE', bbox_to_anchor=(1.02, 1), loc='upper left')

    # Adjust spacing between subplots
    plt.subplots_adjust(bottom=0.2, left=0.1, right=0.9, top=0.9)
    plt.tight_layout()
    plt.savefig(f'../figures/percent_by_occupation_{benefit}_even_sample.png')
    # Show the plot
    plt.show()

In [ ]:
# set font size
plt.rcParams.update({'font.size': 14})

In [ ]:
for df, benefit in zip(benefit_percent_dfs, benefits4):
    plot_benefit_occ_percents(df, benefit)

## Full Sample

In [ ]:
full_sample = pd.read_parquet('../data/us_10m_nointernship_ai_skills_benefits.parquet.gzip')

In [ ]:
full_sample = full_sample[full_sample[occupation].isin(occupations_select)]

In [ ]:
benefit_percent_dfs = []
for benefit in benefits4:
    occ_ai_benefit_percent = get_benefit_by_occupation(full_sample, benefit)
    benefit_percent_dfs.append(occ_ai_benefit_percent)

In [ ]:
occ_ai_benefit_percent

In [ ]:
def plot_benefit_occ_percents(df, benefit):
    sorted_data = df[df['AI ROLE'] == 'No']
    sorted_data = sorted_data.sort_values(by=f'Percent with {benefit}', ascending=True)

    # Create a figure and axis object
    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot the data as a clustered bar chart
    sns.barplot(x=f'Percent with {benefit}', y='Occupation', hue='AI ROLE', data=df, order=sorted_data['Occupation'], ax=ax)

    # Set the labels and title
    ax.set_xlabel(f'Percent Jobs with {benefits_labels_map[benefit]}', fontsize=14)
    ax.set_ylabel('Occupation', fontsize=14)
    ax.set_title(None)

    # Add value labels without decimals
    # for p in ax.patches:
    #     width = p.get_width()
    #     ax.annotate(f'{width:.1f}%', (p.get_x() + p.get_width(), p.get_y() + p.get_height() / 2.),
    #                 ha='center', va='center', fontsize=12, color='black', xytext=(5, 0),
    #                 textcoords='offset points')

    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45)

    # Adjust the legend position and title
    plt.legend(title='AI ROLE', bbox_to_anchor=(1.02, 1), loc='upper left')

    # Adjust spacing between subplots
    plt.subplots_adjust(bottom=0.2, left=0.1, right=0.9, top=0.9)
    plt.tight_layout()
    plt.savefig(f'../figures/percent_by_occupation_{benefit}_sorted.png')
    # Show the plot
    plt.show()

In [ ]:
# set font size
plt.rcParams.update({'font.size': 14})

In [ ]:
for df, benefit in zip(benefit_percent_dfs, benefits4):
    plot_benefit_occ_percents(df, benefit)

# AI Duration

In [ ]:
full_sample['DURATION_CALC'] = pd.to_timedelta(full_sample['DURATION_CALC']).dt.total_seconds() / (24 * 3600)

In [ ]:
duration_df = full_sample.groupby([occupation, 'AI ROLE'])['DURATION_CALC'].mean().reset_index()

In [ ]:
duration_df

In [ ]:
sorted_data = duration_df[duration_df['AI ROLE'] == True]
sorted_data = sorted_data.sort_values(by=f'DURATION_CALC', ascending=True)

# Create a figure and axis object
fig, ax = plt.subplots(figsize=(12, 6))

# Plot the data as a clustered bar chart
sns.barplot(x=f'DURATION_CALC', y=occupation, hue='AI ROLE', data=duration_df, order=sorted_data[occupation], ax=ax)

# Set the labels and title
ax.set_xlabel(f'Average Job Posting Duration', fontsize=14)
ax.set_ylabel('Occupation', fontsize=14)
ax.set_title(None)

# Add value labels without decimals
# for p in ax.patches:
#     width = p.get_width()
#     ax.annotate(f'{width:.1f}%', (p.get_x() + p.get_width(), p.get_y() + p.get_height() / 2.),
#                 ha='center', va='center', fontsize=12, color='black', xytext=(5, 0),
#                 textcoords='offset points')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Adjust the legend position and title
plt.legend(title='AI ROLE', bbox_to_anchor=(1.02, 1), loc='upper left')

# Adjust spacing between subplots
plt.subplots_adjust(bottom=0.2, left=0.1, right=0.9, top=0.9)
plt.tight_layout()
# plt.savefig(f'../figures/duration_by_occ_ai.png')
# Show the plot
plt.show()

# AI Coefficient Analysis

In [ ]:
results = pd.read_csv('../exports/models_occ_year_results.csv')

In [ ]:
results

In [ ]:
wham_results = pd.read_csv('../exports/models_occ_year_results_remote.csv')

In [ ]:
wham_results

In [ ]:
wham_results.dropna(inplace=True)

In [ ]:
wham_results

In [ ]:
results = pd.concat([results, wham_results])

In [ ]:
results.drop(columns='Unnamed: 0', inplace=True)

In [ ]:
results.to_csv('../exports/models_occ_year_results.csv', index=False)

In [ ]:
results

In [ ]:
occ_year_df

In [ ]:
benefits4

In [ ]:
occ_year_df.columns

In [ ]:
coeff_df = results.merge(occ_year_df[['SOC_2021_2_NAME', 'YEAR','AI ROLE %',
       'AI ROLE % CHANGE', 'PRIOR YEAR % CHANGE', 'LOG_SALARY_ai', 'SALARY_ai',
       'LOG_SALARY_non_ai', 'SALARY_non_ai', 'SALARY_PREMIUM_LOG',
       'SALARY_PREMIUM', 'MEAN SALARY', 'MEAN LOG SALARY']], left_on=['Occupation', 'Year'], right_on = [occupation, 'YEAR'])

In [ ]:
coeff_df

In [ ]:
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan', 'black', 'yellow', 'lime', 'teal', 'magenta']

In [ ]:
for i, benefit in enumerate(benefits4):
    benefit_df = coeff_df[coeff_df['Benefit'] == benefit]
    scatter_df = remove_outliers(benefit_df, f'AI Coefficient')
    correlation = scatter_df[['AI ROLE %', 'AI Coefficient']].corr().iloc[0, 1]
    # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        data=scatter_df, 
        x='AI ROLE %', 
        y='AI Coefficient', 
        hue='Occupation',  # Color by occupation
        palette='Set1',         # Optional: Choose a color palette
        legend=None
    )
    
    # Set axis labels and title
    plt.xlabel('AI Role %')
    plt.ylabel('AI Coefficient')
    plt.title(f'{benefits4_labels[i]}\n Correlation: {correlation:.3f}')
    # plt.legend(title='Occupation', bbox_to_anchor=(1, 1))

    plt.savefig(f'../figures/scatter_coeff_ai_demand_nooutliers_{benefit}.png')
    
    # Show the plot
    plt.show()

In [ ]:
for i, benefit in enumerate(benefits4):
    benefit_df = coeff_df[coeff_df['Benefit'] == benefit]
    correlation = benefit_df[['AI ROLE %', 'AI Coefficient']].corr().iloc[0, 1]
    plt.figure(figsize=(8, 6))
    
    # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
    sns.scatterplot(
        data=coeff_df[coeff_df['Benefit'] == benefit], 
        x='AI ROLE %', 
        y='AI Coefficient', 
        hue='Occupation',  # Color by occupation
        palette='Set1',         # Optional: Choose a color palette
        legend=False
    )
    
    # Set axis labels and title
    plt.xlabel('AI Role %')
    plt.ylabel('AI Coefficient')
    plt.title(f'{benefits4_labels[i]}\n Correlation: {correlation:.3f}')

    plt.savefig(f'../figures/scatter_coeff_ai_demand_{benefit}.png')
    
    # Show the plot
    plt.show()

In [ ]:
x_factors = ['AI ROLE %', 'AI ROLE % CHANGE', 'PRIOR YEAR % CHANGE', 'SALARY_PREMIUM_LOG']

In [ ]:
for factor in x_factors:
    for i, benefit in enumerate(benefits4):
        benefit_df = coeff_df[coeff_df['Benefit'] == benefit]
        scatter_df = remove_outliers(benefit_df, f'AI Coefficient')
        correlation = scatter_df[[factor, 'AI Coefficient']].corr().iloc[0, 1]
        # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
        plt.figure(figsize=(8, 6))
        sns.scatterplot(
            data=scatter_df, 
            x=factor,
            y='AI Coefficient', 
            hue='Occupation',  # Color by occupation
            palette='Set1',         # Optional: Choose a color palette
            legend=None
        )
        
        # Set axis labels and title
        plt.xlabel(factor)
        plt.ylabel('AI Coefficient')
        plt.title(f'{benefits4_labels[i]}\n Correlation: {correlation:.3f}')
        # plt.legend(title='Occupation', bbox_to_anchor=(1, 1))

        plt.savefig(f'../figures/scatter_coeff_{factor}_nooutliers_{benefit}.png')
        
        # Show the plot
        plt.show()

In [ ]:
import pandas as pd

In [ ]:
models_occ_year_results = pd.read_csv('../exports/models_occ_year_results_remote.csv')

In [ ]:
models_occ_year_results

In [ ]:
import pandas as pd
import sys
import os

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)
from package_files.benefits_defns import *
path = '../../data/salary_sample_2018_2023.parquet.gzip'
if path[-3:] == 'csv:':
    even_sample = pd.read_csv(path)
elif path[-3:] == 'zip':
    even_sample = pd.read_parquet(path)

occupations_select = ['Architecture and Engineering Occupations','Arts, Design, Entertainment, Sports, and Media Occupations','Business and Financial Operations Occupations','Community and Social Service Occupations','Computer and Mathematical Occupations',
'Educational Instruction and Library Occupations',
'Healthcare Practitioners and Technical Occupations',
'Legal Occupations',
'Life, Physical, and Social Science Occupations',
'Management Occupations', 
'Office and Administrative Support Occupations',
'Personal Care and Service Occupations', 'Production Occupations',
'Sales and Related Occupations',
'Transportation and Material Moving Occupations',
]

even_sample_select = even_sample[even_sample[occupation].isin(occupations_select)]

# get % AI roles per occupation-year
ai_role_occupation = even_sample_select.groupby([occupation, 'YEAR'])['AI ROLE'].mean().reset_index()
ai_role_occupation.rename(columns={'AI ROLE':'AI ROLE %'}, inplace=True)


# # get % with benefit 
# occupation_benefits = even_sample_select.groupby([occupation, 'YEAR', 'AI ROLE'])[benefits3].mean().reset_index()

# benefits_ai = occupation_benefits[occupation_benefits['AI ROLE'] == 1]
# benefits_non_ai = occupation_benefits[occupation_benefits['AI ROLE'] == 0]

# merged_df = pd.merge(benefits_ai, benefits_non_ai, on=['SOC_2021_2_NAME', 'YEAR'], suffixes=('_ai', '_non_ai'))

# salary premium
occupation_salaries = even_sample_select.groupby([occupation, 'YEAR', 'AI ROLE'])[['LOG_SALARY', 'SALARY']].mean().reset_index()
occupation_salaries_ai = occupation_salaries[occupation_salaries['AI ROLE'] == 1]
occupation_salaries_non_ai = occupation_salaries[occupation_salaries['AI ROLE'] == 0]
occupation_salaries = pd.merge(occupation_salaries_ai, occupation_salaries_non_ai, on=[occupation, 'YEAR'], suffixes=('_ai', '_non_ai'))
occupation_salaries['SALARY_PREMIUM_LOG'] = ((occupation_salaries['LOG_SALARY_ai']/occupation_salaries['LOG_SALARY_non_ai'])*100)
occupation_salaries['SALARY_PREMIUM'] = ((occupation_salaries['SALARY_ai']/occupation_salaries['SALARY_non_ai'])*100)

# % change in demand
ai_role_occupation = ai_role_occupation.sort_values(by=[occupation, 'YEAR'])
ai_role_occupation['AI ROLE % CHANGE'] = ai_role_occupation.groupby(occupation)['AI ROLE %'].pct_change()
ai_role_occupation['AI ROLE % CHANGE'] = ai_role_occupation['AI ROLE % CHANGE']*100

# prior year % change
ai_role_occupation['PRIOR YEAR % CHANGE'] = ai_role_occupation.groupby(occupation)['AI ROLE % CHANGE'].shift(1)
occ_year_df = ai_role_occupation.merge(occupation_salaries, on=[occupation, 'YEAR'])

In [ ]:
occupation_salaries

In [ ]:
occ_year_df

In [ ]:
even_sample_select['YEAR']

In [ ]:
# % benefits by occ
occ_year_group = even_sample_select.groupby([occupation, 'YEAR']).size().reset_index(name='job_count')


In [ ]:
occ_year_group

In [ ]:
for benefit in benefits4:
    occ_benefit_group = even_sample_select.groupby([occupation,'YEAR'])[benefit].mean().reset_index(name=f'Percent_with_{benefit}')
    occ_year_group = occ_year_group.merge(occ_benefit_group, on = ['SOC_2021_2_NAME','YEAR'], how = 'left')
    # occ_year_group = occ_year_group.merge(occ_year_benefit_group, on = ['SOC_2021_2_NAME','YEAR'], how = 'left')

In [ ]:
occ_year_group

In [ ]:
results = pd.read_csv('../exports/models_occ_year_results.csv')
# add coefficients
coeff_df = results.merge(occ_year_df[['SOC_2021_2_NAME', 'YEAR','AI ROLE %',
       'AI ROLE % CHANGE', 'PRIOR YEAR % CHANGE', 'LOG_SALARY_ai', 'SALARY_ai',
       'LOG_SALARY_non_ai', 'SALARY_non_ai', 'SALARY_PREMIUM_LOG',
       'SALARY_PREMIUM', 'MEAN SALARY', 'MEAN LOG SALARY']], left_on=['Occupation', 'Year'], right_on = [occupation, 'YEAR'])



In [ ]:
coeff_df = coeff_df.merge(occ_year_group, on = ['SOC_2021_2_NAME','YEAR'])

In [ ]:
even_sample_select

# New Scatterplots

In [ ]:
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    # Define outlier boundaries
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

In [ ]:
coeff_df = pd.read_csv('../exports/occ_year_coeff_analysis.csv')

In [ ]:
x_factors = ['AI Demand', 'AI Demand % Change', 'PRIOR YEAR % CHANGE', 'Salary (Log) Premium']

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
coeff_df

In [ ]:
for factor in x_factors:
    print(factor)
    for i, benefit in enumerate(benefits4):
        print(benefit)
        benefit_df = coeff_df[coeff_df['Benefit'] == benefit]
        benefit_df = benefit_df.dropna(subset=[factor, 'AI Coefficient'])
        scatter_df = remove_outliers(benefit_df, f'AI Coefficient').copy()

        correlation = scatter_df[[factor, 'AI Coefficient']].corr().iloc[0, 1]
        significance_threshold = 0.05
        # Create a new column 'alpha' to set transparency based on the significance of coefficients
        scatter_df['alpha'] = np.where(scatter_df['P-Value'] < significance_threshold, 1, 0.3)  # Example: Adjust alpha based on p-value
        # print(scatter_df.head())
        # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
        plt.figure(figsize=(8, 6))
        sns.scatterplot(
            data=scatter_df, 
            x=factor,
            y='AI Coefficient', 
            hue='Occupation',  # Color by occupation
            style = 'Year', # Style by year
            palette='tab20', 
            alpha = scatter_df['alpha'], # set transparency for insignificant values
            legend=False
        )
        sns.regplot(
            data=scatter_df, 
            x=factor,
            y='AI Coefficient', 
            scatter=False,     # We already have the scatter plot, so only show the trendline
            color='gray',      # Optional: Set the color of the trendline
            line_kws={'linewidth': 2},  # Customize the trendline
        )        
        # Set axis labels and title
        plt.xlabel(factor)
        plt.ylabel('AI Coefficient')
        plt.title(f'{benefits4_labels[i]}\n Correlation: {correlation:.3f}')
        # plt.legend(title='Occupation', bbox_to_anchor=(1, 1))

        plt.savefig(f'../figures/coefficient scatterplots/with trendline/scatter_coeff_{factor}_nooutliers_newcolors_trendline_{benefit}.png')
        
        # Show the plot
        plt.show()

# Single Scatterplot

In [ ]:
for factor in x_factors:
    print(factor)
        benefit_df = benefit_df.dropna(subset=[factor, 'AI Coefficient'])
        scatter_df = remove_outliers(benefit_df, f'AI Coefficient').copy()

        correlation = scatter_df[[factor, 'AI Coefficient']].corr().iloc[0, 1]
        significance_threshold = 0.05
        # Create a new column 'alpha' to set transparency based on the significance of coefficients
        scatter_df['alpha'] = np.where(scatter_df['P-Value'] < significance_threshold, 1, 0.3)  # Example: Adjust alpha based on p-value
        # print(scatter_df.head())
        # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
        plt.figure(figsize=(8, 6))
        sns.scatterplot(
            data=scatter_df, 
            x=factor,
            y='AI Coefficient', 
            hue='Occupation',  # Color by occupation
            style = 'Year', # Style by year
            palette='tab20', 
            alpha = scatter_df['alpha'], # set transparency for insignificant values
            legend=False
        )
        sns.regplot(
            data=scatter_df, 
            x=factor,
            y='AI Coefficient', 
            scatter=False,     # We already have the scatter plot, so only show the trendline
            color='gray',      # Optional: Set the color of the trendline
            line_kws={'linewidth': 2},  # Customize the trendline
        )        
        # Set axis labels and title
        plt.xlabel(factor)
        plt.ylabel('AI Coefficient')
        plt.title(f'{benefits4_labels[i]}\n Correlation: {correlation:.3f}')
        # plt.legend(title='Occupation', bbox_to_anchor=(1, 1))

        plt.savefig(f'../figures/coefficient scatterplots/with trendline/scatter_coeff_{factor}_nooutliers_newcolors_trendline_{benefit}.png')
        
        # Show the plot
        plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set the colors for different perks
palette = sns.color_palette("tab10", len(benefits4))  # Assign different colors for each perk

plt.figure(figsize=(10, 8))  # Set the overall figure size

for factor in x_factors:
    # Loop through each perk (benefit) and plot the scatterplots in the same figure
    for i, benefit in enumerate(benefits4):
        benefit_df = coeff_df[coeff_df['Benefit'] == benefit]
        benefit_df = benefit_df.dropna(subset=[factor, 'AI Coefficient'])
        
        # Remove outliers if needed
        scatter_df = remove_outliers(benefit_df, 'AI Coefficient').copy()

        # Plot scatter points for this perk
        sns.scatterplot(
            data=scatter_df,
            x=factor,
            y='AI Coefficient',
            color=palette[i],  # Use a specific color for this perk
            label=benefits4_labels[i],  # Label for the legend
            alpha=0.7      # Set some transparency for better visibility
        )

        # Plot a fitted line (regression) for this perk
        sns.regplot(
            data=scatter_df,
            x=factor,
            y='AI Coefficient',
            scatter=False,      # No scatter points for the regression line
            color=palette[i],   # Set the color to match the perk
            line_kws={'label': benefits4_labels[i], 'linewidth': 2}  # Add the perk label to the fitted line
        )

    # Add labels and title to the combined plot
    plt.xlabel(factor)
    plt.ylabel('AI Coefficient')
    plt.title('Correlation between AI Demand and AI Coefficient for Different Benefits')

    # Add a legend that shows the perks
    plt.legend(title='Perk', bbox_to_anchor=(1.05, 1), loc='upper left')

    # Display the final combined plot
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set the colors for different perks
palette = sns.color_palette("tab10", len(benefits4))  # Assign different colors for each perk

plt.figure(figsize=(10, 8))  # Set the overall figure size

for factor in x_factors:
    # Initialize variables to track min and max x values across all perks
    min_x = float('inf')
    max_x = float('-inf')

    # First pass: Find the global min and max of the factor for consistent x-axis limits
    for benefit in benefits4:
        benefit_df = coeff_df[coeff_df['Benefit'] == benefit]
        benefit_df = benefit_df.dropna(subset=[factor, 'AI Coefficient'])
        
        # Update global min and max
        if len(benefit_df) > 0:
            min_x = min(min_x, benefit_df[factor].min())
            max_x = max(max_x, benefit_df[factor].max())

    # Second pass: Plot the scatter points and regression lines for each benefit
    for i, benefit in enumerate(benefits4):
        benefit_df = coeff_df[coeff_df['Benefit'] == benefit]
        benefit_df = benefit_df.dropna(subset=[factor, 'AI Coefficient'])
        
        # Remove outliers if needed
        scatter_df = remove_outliers(benefit_df, 'AI Coefficient').copy()

        # Plot scatter points for this perk
        sns.scatterplot(
            data=scatter_df,
            x=factor,
            y='AI Coefficient',
            color=palette[i],  # Use a specific color for this perk
            label=benefits4_labels[i],  # Label for the legend
            alpha=0.7      # Set some transparency for better visibility
        )

        # Plot a fitted line (regression) for this perk
        sns.regplot(
            data=scatter_df,
            x=factor,
            y='AI Coefficient',
            scatter=False,      # No scatter points for the regression line
            color=palette[i],   # Set the color to match the perk
            line_kws={'label': benefits4_labels[i], 'linewidth': 2}  # Add the perk label to the fitted line
        )

    # Set consistent x-axis limits across all perks
    plt.xlim(min_x, max_x)

    # Add labels and title to the combined plot
    plt.xlabel(factor)
    plt.ylabel('AI Coefficient')
    plt.title(f'Correlation between {factor} and AI Coefficient for Different Benefits')

    # Add a legend that shows the perks
    plt.legend(title='Perk', bbox_to_anchor=(1.05, 1), loc='upper left')

    # Display the final combined plot
    plt.tight_layout()
    plt.show()


# Scatterplots v3

In [ ]:
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    # Define outlier boundaries
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

In [ ]:
coeff_df = pd.read_csv('../exports/occ_year_coeff_analysis.csv')

In [ ]:
x_factors = ['AI Demand', 'AI Demand % Change', 'PRIOR YEAR % CHANGE', 'Salary (Log) Premium']

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
coeff_df

In [ ]:
coeff_df['Log Job Count'] = coeff_df['job_count'].apply(lambda x: np.log(x))

In [ ]:
for i, benefit in enumerate(benefits4):
    print(benefit)
    benefit_df = coeff_df[coeff_df['Benefit'] == benefit]
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence', 'LOG_SALARY_ai':'Mean Log AI Salary'}, inplace=True)

    label = benefits_labels_map[benefit]

    x_factors = [f'Overall Benefit Prevalence', 'AI Demand', 'Log Job Count', 'Mean Log AI Salary']
    y_factor = f'Prevalence: {label} (AI)'
    
    for factor in x_factors:
        print(factor)


        benefit_df = benefit_df.dropna(subset=[factor, y_factor])
        scatter_df = remove_outliers(benefit_df, y_factor).copy()

        correlation = scatter_df[[factor, y_factor]].corr().iloc[0, 1]
        significance_threshold = 0.05
        # Create a new column 'alpha' to set transparency based on the significance of coefficients
        scatter_df['alpha'] = np.where(scatter_df['P-Value'] < significance_threshold, 1, 0.3)  # Example: Adjust alpha based on p-value
        # print(scatter_df.head())
        # Create a scatterplot with color mapped to 'SOC_2021_2_NAME' (occupation)
        plt.figure(figsize=(8, 6))
        sns.scatterplot(
            data=scatter_df, 
            x=factor,
            y=y_factor, 
            hue='Occupation',  # Color by occupation
            style = 'Year', # Style by year
            palette='tab20', 
            alpha = scatter_df['alpha'], # set transparency for insignificant values
            legend=False
        )
        sns.regplot(
            data=scatter_df, 
            x=factor,
            y=y_factor, 
            scatter=False,     # We already have the scatter plot, so only show the trendline
            color='gray',      # Optional: Set the color of the trendline
            line_kws={'linewidth': 2},  # Customize the trendline
        )        
        # Set axis labels and title
        plt.xlabel(factor)
        plt.ylabel(y_factor)
        plt.title(f'{benefits4_labels[i]}\n Correlation: {correlation:.3f}')
        # plt.legend(title='Occupation', bbox_to_anchor=(1, 1))

        plt.savefig(f'../figures/coefficient scatterplots/with trendline/scatter_coeff_{factor}_v3_{benefit}.png')
        
        # Show the plot
        plt.show()